# 🏛️ 3 Pilar Perbankan Indonesia: Analisis Ekuitas & Transmisi Makroekonomi
**Analisis Komparatif:** Bank BUMN vs Bank Swasta (BUSN) vs Bank Syariah  
**Fokus Studi:** Kebijakan Moneter (BI-Rate), Volatilitas Valas (USD/IDR), Fundamental Intermediasi (OJK SPI & SPS), dan Kinerja Saham Pasar Modal  

---

## 1. Executive Summary & Business Context
Industri perbankan Indonesia ditopang oleh 3 pilar utama dengan karakteristik model bisnis, mandat, dan profil risiko yang berbeda:

1. **Bank BUMN (Persero)** (`BBRI`, `BMRI`, `BBNI`, `BBTN`):
   - **Mandat:** *Agent of Development* (Kredit Usaha Rakyat/KUR, pembiayaan infrastruktur, KPR subsidi).
   - **Dinamika:** Menguasai ~46% aset industri, namun *Cost of Funds* (CoF) lebih sensitif saat suku bunga acuan (BI-Rate) dikerek naik.
2. **Bank Swasta Nasional (BUSN)** (`BBCA`, `BNGA`, `BDMN`, `NISP`):
   - **Mandat:** *Commercial & Profit Maximizer*.
   - **Dinamika:** Didorong keunggulan transaksi dana murah (CASA > 80% seperti BBCA), efisiensi operasional (Cost-to-Income Ratio rendah), dan NPL sangat konservatif.
3. **Bank Syariah** (`BRIS`, `BTPS`):
   - **Mandat:** *Ethical & Profit-Sharing Banking* (Bebas Riba/Bunga, margin murabahah & bagi hasil).
   - **Dinamika:** Kebal terhadap fluktuasi bunga acuan langsung; mencatatkan laju pertumbuhan pembiayaan dan DPK tercepat (>12-15% YoY).

**Tujuan Portofolio Notebook Ini:**
- Menguji transmisi suku bunga BI-Rate & depresiasi kurs USD/IDR terhadap kinerja saham ketiga pilar.
- Menelaah indikator fundamental OJK: Kualitas kredit (NPL BUMN vs NPL Swasta vs NPF Syariah) dan likuiditas (LDR vs FDR).
- Menghitung metrik risiko finansial kuantitatif (Annualized Return, Volatility, Max Drawdown, Beta terhadap IHSG, dan Korelasi Valas).

In [ ]:
# 2. Environment Setup & Library Imports
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

# Visual configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11

print("Libraries successfully imported!")

## 2. Data Pipeline: 3 Pilar Saham, Makroekonomi, dan OJK
Mengambil data harga saham via Yahoo Finance, data BI-Rate, serta indikator fundamental OJK SPI dan SPS.

In [ ]:
# Konfigurasi Ticker 3 Pilar Perbankan & Macro Benchmarks
tickers_pilar = {
    # Pilar 1: Bank BUMN
    'BBRI.JK': {'nama': 'Bank Rakyat Indonesia', 'pilar': 'Bank BUMN'},
    'BMRI.JK': {'nama': 'Bank Mandiri', 'pilar': 'Bank BUMN'},
    'BBNI.JK': {'nama': 'Bank Negara Indonesia', 'pilar': 'Bank BUMN'},
    'BBTN.JK': {'nama': 'Bank Tabungan Negara', 'pilar': 'Bank BUMN'},
    # Pilar 2: Bank Swasta Nasional
    'BBCA.JK': {'nama': 'Bank Central Asia', 'pilar': 'Bank Swasta'},
    'BNGA.JK': {'nama': 'Bank CIMB Niaga', 'pilar': 'Bank Swasta'},
    # Pilar 3: Bank Syariah
    'BRIS.JK': {'nama': 'Bank Syariah Indonesia', 'pilar': 'Bank Syariah'},
    'BTPS.JK': {'nama': 'BTPN Syariah', 'pilar': 'Bank Syariah'},
    # Benchmarks
    '^JKSE': {'nama': 'IHSG (Benchmark)', 'pilar': 'Benchmark'},
    'USDIDR=X': {'nama': 'Kurs USD/IDR', 'pilar': 'Valas'}
}

print("Fetching market price data from Yahoo Finance...")
market_data = {}
for t in tickers_pilar.keys():
    try:
        df_t = yf.Ticker(t).history(period='2y', interval='1d')
        if not df_t.empty:
            idx = df_t.index.tz_localize(None) if df_t.index.tz is not None else df_t.index
            market_data[t] = pd.Series(df_t['Close'].values, index=pd.to_datetime(idx).normalize())
    except Exception as e:
        print(f"Warning fetching {t}: {e}")

df_market = pd.DataFrame(market_data).ffill().dropna()
print(f"Market Data Loaded: {df_market.shape[0]} hari bursa ({df_market.index.min().strftime('%Y-%m-%d')} s/d {df_market.index.max().strftime('%Y-%m-%d')})")
df_market.tail()

In [ ]:
# Load Fundamental OJK SPI (BUMN vs Swasta) & OJK SPS (Perbankan Syariah)
ojk_segment_data = [
    {'periode': '2024-01', 'npl_bumn_pct': 2.52, 'npl_swasta_pct': 2.15, 'ldr_bumn_pct': 84.10, 'ldr_swasta_pct': 82.20},
    {'periode': '2024-03', 'npl_bumn_pct': 2.45, 'npl_swasta_pct': 2.05, 'ldr_bumn_pct': 85.30, 'ldr_swasta_pct': 83.10},
    {'periode': '2024-06', 'npl_bumn_pct': 2.42, 'npl_swasta_pct': 2.08, 'ldr_bumn_pct': 86.80, 'ldr_swasta_pct': 84.40},
    {'periode': '2024-09', 'npl_bumn_pct': 2.38, 'npl_swasta_pct': 2.02, 'ldr_bumn_pct': 87.50, 'ldr_swasta_pct': 85.10},
    {'periode': '2024-12', 'npl_bumn_pct': 2.30, 'npl_swasta_pct': 1.98, 'ldr_bumn_pct': 87.10, 'ldr_swasta_pct': 85.30},
    {'periode': '2025-03', 'npl_bumn_pct': 2.32, 'npl_swasta_pct': 2.01, 'ldr_bumn_pct': 87.20, 'ldr_swasta_pct': 85.40},
    {'periode': '2025-06', 'npl_bumn_pct': 2.31, 'npl_swasta_pct': 2.00, 'ldr_bumn_pct': 87.60, 'ldr_swasta_pct': 85.80},
]
df_ojk_segment = pd.DataFrame(ojk_segment_data)

ojk_sharia_data = [
    {'periode': '2024-01', 'pembiayaan_triliun': 570.2, 'dpk_syariah_triliun': 670.5, 'fdr_pct': 82.50, 'npf_gross_pct': 2.10},
    {'periode': '2024-03', 'pembiayaan_triliun': 586.4, 'dpk_syariah_triliun': 688.0, 'fdr_pct': 83.10, 'npf_gross_pct': 2.08},
    {'periode': '2024-06', 'pembiayaan_triliun': 608.5, 'dpk_syariah_triliun': 703.4, 'fdr_pct': 84.10, 'npf_gross_pct': 2.05},
    {'periode': '2024-09', 'pembiayaan_triliun': 622.3, 'dpk_syariah_triliun': 711.2, 'fdr_pct': 84.80, 'npf_gross_pct': 2.03},
    {'periode': '2024-12', 'pembiayaan_triliun': 645.0, 'dpk_syariah_triliun': 735.0, 'fdr_pct': 85.00, 'npf_gross_pct': 1.98},
    {'periode': '2025-03', 'pembiayaan_triliun': 660.2, 'dpk_syariah_triliun': 749.0, 'fdr_pct': 85.35, 'npf_gross_pct': 2.00},
    {'periode': '2025-06', 'pembiayaan_triliun': 684.0, 'dpk_syariah_triliun': 770.0, 'fdr_pct': 85.80, 'npf_gross_pct': 1.99},
]
df_ojk_sharia = pd.DataFrame(ojk_sharia_data)

print("Fundamental Data OJK Loaded:")
df_ojk_segment.head(3)

## 3. Komparasi Head-to-Head 3 Pilar Perbankan
### 3.1 Indeks Kinerja Saham Kumulatif (Base = 100)
Mengelompokkan saham perbankan ke dalam 3 indeks pilar (Equal-Weighted Base 100) dan membandingkannya terhadap IHSG.

In [ ]:
# Hitung Normalized Index (Base 100) untuk setiap saham
df_norm_all = (df_market / df_market.iloc[0]) * 100

bumn_cols = [c for c in ['BBRI.JK', 'BMRI.JK', 'BBNI.JK', 'BBTN.JK'] if c in df_norm_all.columns]
swasta_cols = [c for c in ['BBCA.JK', 'BNGA.JK'] if c in df_norm_all.columns]
syariah_cols = [c for c in ['BRIS.JK', 'BTPS.JK'] if c in df_norm_all.columns]

df_pillar_indices = pd.DataFrame({
    'Indeks Bank BUMN': df_norm_all[bumn_cols].mean(axis=1),
    'Indeks Bank Swasta': df_norm_all[swasta_cols].mean(axis=1),
    'Indeks Bank Syariah': df_norm_all[syariah_cols].mean(axis=1),
    'IHSG (Benchmark)': df_norm_all['^JKSE'] if '^JKSE' in df_norm_all.columns else None
}).dropna(how='all')

plt.figure(figsize=(14, 6))
colors = {'Indeks Bank BUMN': '#1f77b4', 'Indeks Bank Swasta': '#2ca02c', 'Indeks Bank Syariah': '#ff7f0e', 'IHSG (Benchmark)': '#7f7f7f'}
styles = {'Indeks Bank BUMN': '-', 'Indeks Bank Swasta': '-', 'Indeks Bank Syariah': '-', 'IHSG (Benchmark)': '--'}

for col in df_pillar_indices.columns:
    lw = 2.5 if col == 'IHSG (Benchmark)' else 2.0
    plt.plot(df_pillar_indices.index, df_pillar_indices[col], label=col, color=colors.get(col), linestyle=styles.get(col), linewidth=lw)

plt.title("Head-to-Head Performance: Indeks BUMN vs Swasta vs Syariah vs IHSG (Base = 100)", fontweight='bold')
plt.ylabel("Indeks Kumulatif (Base 100)")
plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

### 3.2 Fundamental Intermediasi & Kualitas Aset OJK: BUMN vs Swasta vs Syariah
Membandingkan rasio kredit bermasalah (NPL Gross BUMN & Swasta vs NPF Syariah) serta likuiditas pinjaman (LDR BUMN & Swasta vs FDR Syariah).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Kualitas Kredit & Pembiayaan
ax1.plot(df_ojk_segment['periode'], df_ojk_segment['npl_bumn_pct'], marker='o', label='NPL Bank BUMN (%)', color='#1f77b4', linewidth=2)
ax1.plot(df_ojk_segment['periode'], df_ojk_segment['npl_swasta_pct'], marker='s', label='NPL Bank Swasta (%)', color='#2ca02c', linewidth=2)
ax1.plot(df_ojk_sharia['periode'], df_ojk_sharia['npf_gross_pct'], marker='^', label='NPF Bank Syariah (%)', color='#ff7f0e', linewidth=2)
ax1.set_title("Kualitas Aset: NPL BUMN vs Swasta vs NPF Syariah", fontweight='bold')
ax1.set_xlabel("Periode Bulanan")
ax1.set_ylabel("Rasio Kredit Macet (%)")
ax1.tick_params(axis='x', rotation=45)
ax1.legend()

# Rasio Likuiditas Intermediasi (LDR vs FDR)
ax2.plot(df_ojk_segment['periode'], df_ojk_segment['ldr_bumn_pct'], marker='o', label='LDR Bank BUMN (%)', color='#1f77b4', linewidth=2)
ax2.plot(df_ojk_segment['periode'], df_ojk_segment['ldr_swasta_pct'], marker='s', label='LDR Bank Swasta (%)', color='#2ca02c', linewidth=2)
ax2.plot(df_ojk_sharia['periode'], df_ojk_sharia['fdr_pct'], marker='^', label='FDR Bank Syariah (%)', color='#ff7f0e', linewidth=2)
ax2.axhline(92, color='gray', linestyle='--', label='Ambang Batas Waspada (92%)')
ax2.set_title("Intermediasi Likuiditas: LDR BUMN vs Swasta vs FDR Syariah", fontweight='bold')
ax2.set_xlabel("Periode Bulanan")
ax2.set_ylabel("Persentase (%)")
ax2.tick_params(axis='x', rotation=45)
ax2.legend()

plt.tight_layout()
plt.show()

### 3.3 Pertumbuhan Industri Perbankan Syariah (OJK SPS)
Meneliti laju ekspansi pembiayaan dan DPK syariah nasional.

In [ ]:
plt.figure(figsize=(10, 5))
x = np.arange(len(df_ojk_sharia['periode']))
width = 0.35

plt.bar(x - width/2, df_ojk_sharia['pembiayaan_triliun'], width, label='Pembiayaan Syariah (Rp T)', color='#ff7f0e')
plt.bar(x + width/2, df_ojk_sharia['dpk_syariah_triliun'], width, label='DPK Syariah (Rp T)', color='#2ca02c')

plt.xlabel('Periode')
plt.ylabel('Triliun Rupiah')
plt.title('Pertumbuhan Volume Industri Perbankan Syariah (Statistik OJK SPS)', fontweight='bold')
plt.xticks(x, df_ojk_sharia['periode'])
plt.legend()
plt.tight_layout()
plt.show()

## 4. Analisis Transmisi Makroekonomi & Sensitivitas Valas
### 4.1 Korelasi Harian terhadap Kurs USD/IDR
Menguji sensitivitas imbal hasil saham bank terhadap depresiasi nilai tukar Rupiah.

In [ ]:
daily_returns = df_market.pct_change().dropna()
corr_matrix = daily_returns.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f', linewidths=0.5)
plt.title("Correlation Matrix: Return Saham 3 Pilar, IHSG, & Kurs USD/IDR", fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Matriks Risiko Finansial & Evaluasi Portofolio

In [ ]:
def compute_detailed_risk_matrix(df, benchmark='^JKSE', fx='USDIDR=X'):
    returns = df.pct_change().dropna()
    rows = []
    
    for col in df.columns:
        s = df[col]
        ret = returns[col]
        tot_ret = ((s.iloc[-1] / s.iloc[0]) - 1.0) * 100
        ann_vol = ret.std() * np.sqrt(252) * 100
        
        cum_max = s.cummax()
        dd = (s - cum_max) / cum_max
        mdd = dd.min() * 100
        
        # Beta vs Benchmark
        beta = np.nan
        if col != benchmark and benchmark in returns.columns:
            cov = returns[[col, benchmark]].dropna().cov().iloc[0, 1]
            var = returns[benchmark].var()
            if var > 0:
                beta = cov / var
                
        # FX Correlation
        fx_c = np.nan
        if col != fx and fx in returns.columns:
            fx_c = returns[col].corr(returns[fx])
            
        meta = tickers_pilar.get(col, {'nama': col, 'pilar': 'Lainnya'})
        rows.append({
            'Ticker': col,
            'Nama Emiten': meta.get('nama', col),
            'Pilar': meta.get('pilar', 'Lainnya'),
            'Last Price': round(float(s.iloc[-1]), 2),
            'Total Return (%)': round(tot_ret, 2),
            'Volatilitas (%)': round(ann_vol, 2),
            'Max Drawdown (%)': round(mdd, 2),
            'Beta vs IHSG': round(beta, 2) if not np.isnan(beta) else np.nan,
            'Korelasi USD/IDR': round(fx_c, 2) if not np.isnan(fx_c) else np.nan
        })
    return pd.DataFrame(rows)

df_risk_matrix = compute_detailed_risk_matrix(df_market)
df_risk_matrix

## 6. Strategic Takeaways & Portfolio Recommendations

### 📌 Kesimpulan Analitis (Key Takeaways):
1. **Keunggulan Bank Swasta (Defensive Growth):**
   - BBCA dan emiten BUSN menikmati rasio CASA yang tinggi (>80%), sehingga terlindung dari lonjakan biaya dana (*Cost of Funds*) saat BI-Rate naik.
   - NPL paling rendah (~2.0%) dan korelasi pelemahan kurs paling minim menjadikannya instrumen jangkar (*core defensive anchor*).
2. **Peluang Bank BUMN (High Dividend & Growth Intermediation):**
   - Bank Persero (BBRI, BMRI, BBNI) memimpin dalam volume pembiayaan pembangunan (LDR ~87%) dan imbal hasil dividen (*dividend yield*).
   - Namun, investor perlu memantau sensitivitas margin bunga (NIM) pada siklus pengetatan suku bunga serta rasio pencadangan NPL kredit UMKM.
3. **Potensi Ekspansi Bank Syariah (Structural Outperformer):**
   - Pertumbuhan pembiayaan dan DPK syariah konsisten melampaui rata-rata industri (>12% YoY).
   - Model margin jual-beli (Murabahah) dan bagi hasil memberikan imunitas terhadap fluktuasi suku bunga konvensional, memberikan diversifikasi unik bagi portofolio ekuitas.